# PhiState CPU binding journal

Select **Runtime → Change runtime type → CPU**, then run the single code cell. When Colab prompts you, mount the Drive account containing `MyDrive/Quantonium/PhiState-L4`.

This journal tests one skill. Given two named values, can the model return the requested one when the question changes or the values trade places? A structured reference learns from explicitly parsed records, and a separate dense reference gets tokenized text. If the structured reference passes, the journal fits a separate copy of your frozen SFT-4,000 checkpoint using a fresh optimizer for up to **96 CPU updates**. It scores all four combinations of query and value assignment on training examples, fresh bindings, and fresh wording.

If the warm model misses its gate, the journal automatically tries single-value copying for up to **160 additional CPU updates**, with early stopping. If copying passes, a final **96-update two-value experiment** starts from the verified copy checkpoint and measures whether the model can now select the requested value. These checks measure a narrow retrieval skill. Allow **2 GiB of free Drive space** for the stage snapshots and the temporary checkpoint used during saves. The cell streams progress and saves its report to `whitespace_repair_v1/phistate_cpu_binding_v1/report.json`.

**After a disconnect**, reconnect to CPU and run the same cell. Only enable `RECOVER_DISCONNECTED` once the previous runtime has ended and a stale run lock is blocking recovery. CPU runtime varies with the assigned Colab machine.

In [ ]:
#@title Run PhiState binding test on CPU
RECOVER_DISCONNECTED = False #@param {type:"boolean"}

import base64
import hashlib
import importlib.util
import json
import os
import re
import shutil
import signal
import subprocess
import sys
import tempfile
import zlib
from pathlib import Path

PIN = "73edfa1a704e8d198269351ebe597e5ff8fed8ed"
SOURCE_HASH = "dcc79a5a3b05411725fefcefa04fd0aafd4393f4efb21e018c60da1e9a80d8ea"
REPO = "https://github.com/Anharmoniclabs/quantoniumoslinux.git"
LAB = Path("runtime/experiments/phi_runtime_lab")
MOUNT = Path("/content/drive")
ROOT = MOUNT / "MyDrive/Quantonium/PhiState-L4/whitespace_repair_v1"
ENTRY = "phistate_binding_journal.py"
REPORT = ROOT / "phistate_cpu_binding_v1/report.json"

def require_cpu():
    nvidia = shutil.which("nvidia-smi")
    if nvidia:
        info = subprocess.run([nvidia, "-L"], capture_output=True, text=True, timeout=15)
        if info.returncode == 0 and re.search(r"(?m)^GPU \d+:", info.stdout):
            raise RuntimeError("A GPU is allocated. Change runtime type to CPU, reconnect and run this cell.")
    if importlib.util.find_spec("torch") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                        "--index-url", "https://download.pytorch.org/whl/cpu", "torch"], check=True)
    import torch
    if torch.cuda.is_available():
        raise RuntimeError("A CUDA device is attached. Change runtime type to CPU, reconnect and run this cell.")
    print("Verified CPU runtime | PyTorch", torch.__version__, flush=True)

def verify_checkpoint():
    checkpoint = ROOT / "coherence_execution_v1/source.pt"
    if not checkpoint.is_file():
        raise FileNotFoundError(f"Frozen SFT-4,000 checkpoint missing from the mounted Drive: {checkpoint}")
    with checkpoint.open("rb") as stream:
        checksum = hashlib.file_digest(stream, "sha256").hexdigest()
    if checksum != SOURCE_HASH:
        raise RuntimeError(f"Unexpected frozen checkpoint SHA-256: {checksum}")
    print("Verified SFT-4,000 checkpoint |", checksum, flush=True)

def checkout(stage):
    def git(*args):
        return subprocess.check_output(["git", "-C", str(stage), *args], text=True).strip()
    git("init", "-q")
    git("remote", "add", "origin", REPO)
    git("config", "remote.origin.promisor", "true")
    git("config", "remote.origin.partialclonefilter", "blob:none")
    git("sparse-checkout", "set", "--no-cone", "/runtime/experiments/phi_runtime_lab/", "/runtime/src/rftmw_native/")
    git("fetch", "--quiet", "--depth=1", "--filter=blob:none", "origin", PIN)
    git("checkout", "--quiet", "--detach", "FETCH_HEAD")
    lab = stage / LAB
    if git("rev-parse", "HEAD") != PIN or not (lab / "moe_train.py").is_file():
        raise RuntimeError("Public PhiState checkout could not be verified.")
    return lab

def unpack_scripts(lab):
    packed = zlib.decompress(base64.b64decode(SOURCES, validate=True))
    if hashlib.sha256(packed).hexdigest() != PACKAGE_SHA:
        raise RuntimeError("Embedded experiment package did not pass its integrity check.")
    modules = json.loads(packed)
    if ENTRY not in modules:
        raise RuntimeError("Embedded experiment entry point missing.")
    for name, source in modules.items():
        if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.py", name):
            raise RuntimeError("Invalid embedded module filename.")
        compile(source, name, "exec")
        with (lab / name).open("xb") as stream:
            stream.write(source.encode("utf-8"))

def run_worker(lab):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="", PYTHONUNBUFFERED="1", PYTHONDONTWRITEBYTECODE="1",
               TOKENIZERS_PARALLELISM="false", PHISTATE_JOURNAL_COMMIT=PIN,
               PYTHONPATH=os.pathsep.join([str(lab), *filter(None, [os.environ.get("PYTHONPATH")])]))
    args = [sys.executable, "-u", str(lab / ENTRY), "--root", str(ROOT)]
    if RECOVER_DISCONNECTED:
        args.append("--recover-disconnected")
    interrupted = False
    with subprocess.Popen(args, cwd=lab, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace", bufsize=1, start_new_session=True) as child:
        with child.stdout:
            while True:
                try:
                    for line in child.stdout:
                        print(line, end="", flush=True)
                    code = child.wait()
                    break
                except KeyboardInterrupt:
                    if not interrupted:
                        interrupted = True
                        print("Stop requested. Waiting for the worker to exit.", flush=True)
                        try:
                            child.send_signal(signal.SIGINT)
                        except ProcessLookupError:
                            pass
    if code and not (interrupted and code in (-signal.SIGINT, 130)):
        raise RuntimeError(f"Binding worker exited with status {code}. See the last worker message above.")
    return interrupted

def launch():
    require_cpu()
    if not (MOUNT / "MyDrive").is_dir():
        from google.colab import drive
        drive.mount(str(MOUNT))
    verify_checkpoint()
    missing = [m for m in ("numpy", "sentencepiece") if importlib.util.find_spec(m) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
    with tempfile.TemporaryDirectory(prefix="phistate_binding_", dir="/content") as tmp:
        lab = checkout(Path(tmp))
        unpack_scripts(lab)
        print("Starting | base", PIN, "| experiment", PACKAGE_SHA[:16], "| report", REPORT, flush=True)
        if run_worker(lab):
            print("Worker stopped. Rerun on CPU to resume.", flush=True)
            return
    if not REPORT.is_file():
        raise RuntimeError(f"Worker ended without its expected Drive report: {REPORT}")
    status = json.loads(REPORT.read_text(encoding="utf-8"))
    print(f"Drive report: {REPORT}", flush=True)
    for key in ("status", "outcome", "conclusion", "next_action"):
        if key in status:
            print(f"{key}: {status[key]}", flush=True)

# Compressed UTF-8 experiment sources; verified before running.
PACKAGE_SHA = "4d3868e583bf3fcc6ac80aaef6f317310b0f46df6f0ffc9a362f2a25cbf9d7a9"
SOURCES = (
    'eNrNfWt320aW4F+B1acHgA3RoizLNmVkNp0ovX0mHWttZ/rsUFwciCxKiEiABkA9Wq2v8wPmJ+4v2fuoKhQKBZJyku1J97EI4Nbr'
    '1q37qlu3HvZWV1lVp7VILrJ8luWXyUWZzS7FYHW/N9o7p/99V+R1lq+FV18J70aU2TwTM68C4IXYnxare296JabXqyLLa6/ICaxK'
    'lwB/W+yXYlqUM29aFlVVQGFvmlaiGnDN53m2XBVl7f1SFfl5Pi+LpbdK66tFduHJL2fweJ5rwErktcinYpWJqfDSyqtWS/2xLsqp'
    'CdwZ2yytUyyEf/uh5lmNQLdpuewC4XiTaQo9LNM6g9ECJL6TvZ9epXWCY4ah3nN7sobPp389+/Hbz6eRV12lTui6TLNcgad1scym'
    'CSIm8sp1niyK6bUstixEGxiQXwNAOsPBn+czMfeqYl1OBVSOAAH1epaV4eg89+A/fh1j7QMsVwUa5KXPHwf40Q8HpUhnSS3u6iAM'
    'uXA29wIJcynqwEfErCs/fBb702K5Woha+F5ReiZMsa7hmyAgphzG5CqtKjHzuWL1n1W2EiJP6qJOF1j81aH9fSZuzM/b6wKKBKTX'
    'fnQQvj9856quBXG0scZSfFmLCkAfHkN+syqLupgW1B1fEQ4tlH2DcPZvhn3DHutKJ7LXNJlYHxLl4NOHnz9+d7oBvi6uRZ79XZS6'
    'yOcP/3b601/+4/SjogCigjSrhPfv6WItTsuyKAP/MyxdZgAeVpmVovJSDycJ5qxZ+7zom7H4oSKrel3mslcNLS5FWq1LnvFgWczE'
    'IloVxaJqqJGKPVyPqK/iBnoEKDNBx9eTaBh6cxjztQeED2PEBZDM02W2yNLSj2jerMdc3PrhY9MRWEfBKi2Bh0QaRQkynEhRfwSE'
    'ulrX0fOoqsWqit8dR0Ays/jw4PB4ODwYRvUVLogqPlSd315fvExXwSJdXswAlSNkaMEK11VVLG5EEEa7dclcffCGkKBBnDUM5Fv+'
    'I59CJJu8qL3h+5iG+D4evuuuGQkhR/s+fruZbn6G5xTYEWB5JeAfEASSjCQ+veFgAO146xXwRCSqfIav3nrfnf3syVb8ZoTAIiVW'
    'ujSP39oD7RD5xr7+rSyAloGR/l3k3qcfPu8fHRwcSH6J9RuLh6u5AIkV9/BThqjKadzwzwWOEPBd+4q8v8SzbFoHijHYXKEu9qXs'
    'QaYQcVPmsJuJtQYatSeOFmZyK7LLq7qKEVHQs5ApSbJ8fNnD6hlOsfBZjOMe+0QkzVt/YrfJC4X+1atD/uW1g/9EC9n1Hz9GX9Zp'
    'WYu6SlYwf0wQ8VFEwhuqKQP6NUgSYDcVMJckCbvDnIn4YTXIQcsYEa0wa1jRqqAVloAYByEDZemR2rbe4XDtdyixm3fho91ysaqz'
    'JU0E4K5aL1kpUgzS1Im+naXLv3lyLojelyAA81qTOayMwfIapkFSehV/LtciEndAGklxTU8S9Darr7QSgKvfXI3LNM/mQHAxvH+p'
    'JAFPaQMEa0rBDaiFKjDrkCCGPqChTfn/LIbqR3JNfVyDWrhUq+pPjdSAUt4yq5ZpPb1Sg8X/xAKaSPP7gCeOeAyx8k+fP5wBx1Zd'
    'B3TUGUrzZk4RV1ktSsRWaHfc1Z2f8wy5EIsryYqgMMj0orxvd6oSI0PZ0gOPoDsGXLEALCjqJgIFSZvk66V6GxjAdXlv9ZGLVFYR'
    '+Tdsw9Jsc4ES6KZYDgAP10mZXwYg1LKpqOLxxEYCM5pqvahjVGB7ZAqwgwiXNcoUa4A7cl5kKEQGY99kNv7E1R/HxHzSfNa2HEAZ'
    'zi9htmbrEtcST5pv9VBrGDjS5tM8y9PFYjekGzMZNqqBhbQedLXVZ7ngGhZ6wqyfP9hiAJArYR0LkM0kUyGXsF31W9eFJQzVjxfn'
    'xrVJxoJraRr1SW1+Ehva/EijHWG42DKqYm13BMi3JdkYUlxqr/M6IfGA2igJW6Ql/kYWUC2gHcAWfldm0qhXcnMzyBtgGGoQoCh6'
    'Ff+E2YvBKhx8kqbiGZqKZ2UB66aCWkipJAYvhQ3Pteo2aZtM+Bdpfl0hgPxE9I4vSdYPUFQAbZkQJNMle2C0AAGB9od8mxRlwOlq'
    '7Udy0SRFvrg3+TziDKtoWxaAFRbFeqoJgQ0gIreB6gjsjdrQmRRdtBolhcwK0NCIOzMPvUchB+XUWpe9XdIYE7aLScGhn2Nf8wQl'
    '6Va1RAqJzwGJxmCJjB7kALB1WAumgsDlk5mYpvfxwSCCqRYpKAc/pMCsmzr7mtcyGjpw0vAA4J8MrODgBUIQ6eAbY5bhGdZ3226J'
    '9PwbmngDPbYtksnYF3fpFMaRXWZA5FdgeYqiwoUqJ8q0RXFO29zLrLpl22yt2LRgt9eLRtLWKgEouYW1j/4RXfVmm+C7Yr2YERkB'
    'WZXFbD1lZYmZCDD6lUl6jSDw9WISq/jgZLoub0DZe9Q4l3zVwUQ7y49Bn7ICzVXY4a2bFJ8pO8nYJ2RINRe33WHhnOxI3+bgAV0S'
    'ghjCRKJOvqMHePnkBYHSsRlQC+G0YKsUrFiJaeqmxBqiTOE5BrBmHGHUqNE4ztanrhrhdcwNsjYiHh79G0HXDbWsGVkYhs0omPIC'
    'lnBt6UuKU6f3DBnzn8hirN1OuLsu2UYF6m8u4mb1RQ1SSRIxCkMHiZOS6v1U5B0qLQPsRIgqMnXB1oy5Y2MNOLG1KRz3gG2wgPiR'
    'NM6SEpdSOfalYYocosvfLstivSLQrlnY+s/kX5qPbGjK5ndPasjkVVvakLxv5+qlJV2jWAReUrIS3FqHpi3B8jRiNIcnDuWVXLV1'
    'sYJlJb8GpDySSRQ2FAACHz9IJ07nu6E+mHVJgvfx7QpUgQbOP/CV/WUTDpMMAEx4QUiUsXSWIhGkIXvwSCgaRI2D5DbBXM0R07JN'
    'Ip6YfXkdSmIQtPbQhwRsFgwBItgXw4hMDVYfJy+G5qpthtu1YgB+1OJaPWaEEz9UyUxqgasMTO6FyAPqdBgdRUG+PwyfH8meCVSx'
    'XgxBrTVKl8UtGGklDSnDIUF19FDiA9U0ziYTUyTQy4AZPzDGIrksUYEE2q2LJIe1z2LqZFFUsmfFxS8ghrMbnA1ssM0ccHaZI2YV'
    'mEdgPQdYNJQWwg8gXEBgXZ4heqQoAw7DkNp5p1owMYO1gG48vYZOzAKlYeX5YF1ni2owXWQr6jt0ulwmtpo3HEQCW0uyOY6K27Ml'
    'MAsFAVN7Quw+b4+MvAYmXbx8ediiE5siGi6Yh7tTda/6ZyH6j8PjOCaj5it6ZpBo39JR/61KBJuDYgX60v/9z/8y99fkcLwHbOfx'
    '5QO2eb5HjZ7vTR4jmjXvYY6zTnQwmIkadGr0pgyO5o9+NF+sqytzIqw1oo1B+LwX7Tn31sxdw+9xxpcwvVWdTT2QqeU9ecFQ6bvM'
    '0RNmbwqif6FEd829d1HUV+QanmXpZV5gFc2eod6UQ49QjRNmbDWk14ImSXMFLnWKO2weSjjvWty/xB0GwZ5er5gCeVQ4c9RqtVpk'
    'dRWRFwb97NgpxGyqcKz7wb7h/NqLPaWe4hKnCiIQFaAmEj0E53u0uM/3Iu/wKIw8eAHCBx+Hxy1vFsmhCuobG6xB8xBmi1SvTUYX'
    '6BiMoI8xYOwGTNwgi7y3Fv1cwNcg9V54wWHkHUXecTimcpPQ+6P3tg17o2APvedcO/xG8qPReXHsyTGEjsK3WPhGNnQMbW1oiIaM'
    'GOyK3fO9bHa+N/Lm53sP1CzQdTY6OJw9Aupc4DCzFRYIpldlcPwaOpACtvXDRRg6ixExcEFkETdQBv/edsAfHV2vBukKtzwCejIw'
    'jpQxpm5PYHgM21pYCNAQLq0AWUmLcj/dpiveeadu0hri1YQ2MVLziYdWDMEs0qrGV6BXTMF4r7zbqxTFgGyzRbppRBRBTY4V8iaq'
    'i6iiGISIRHgDi4JkmVp2gxWs8HVNNo/s+7hBZ4vXYXnqNRBpXt2KMoJX2YLXB/QE68aVcQF/4anr42VNVeK6O4sPbq2NhjXEqU2j'
    'PgjqL8HcRBtqOUSIi821EMxtLwwhAEEYE31gjCDqNKOqDxAxSAsE/jqALHK12ToroYr+pMIcgCIRwcdLMHBjaEGqaLDOR4YXE78y'
    'D2gAWuYUNXG+95FkRzXyHnAaHuMHxvXjCb04VC8OHwfeRy6jCd0r5t4D4Yk+roDAyS8OipBHTgqg5QH7AC1yeP4cxtC8Cp39BsXf'
    '3eUfgFC5t9QXasrLQHTKrg88BXHohMCx/O0qg5XJHy7Rl0Y9l6P5199iNB1vy/meHN1yDRzgQnhqYkgtwcH2iO05tG1I7T+BdEHf'
    'DMjefeIqGAyDxiitYByv3Di9KIt01myfnl1ln7BeQ6GxY3yu0goDe6yYH/nAGx2/WQiQ+YSq6XydT5FLpQss8MN5/oRwHWirmIlE'
    '3KWoAP3O4Ts8fmfckgJGQRE1ik6kli4OSu7VxJ4/m07fvEtfp68uDl4fDYdvDl/PxXwq5unB0Xx2kKbz2dGrd6/mR2J+cTgUB8O3'
    '0+ODWToU79K3B7O3IvXPc73LjPW9m7+BkvPjdzNxPL04PpjP3hwdH6eHxxdvDqYXs4OjN4cXF8N0+O5wPnx7+Dp9M319OJ1PXx0M'
    '37wT0CjU9+nz6dknqOvd8Xn+3Yez/52oF0N48eePH34++5ScnX5Mfj77HhAP74+gzOnp9/BLhV+c5z9ibw4GQHNHDfPiKWqiAiLQ'
    'lUFdrdvCRrEwpScGYLBeijrssoH2lAfjB78sFsIf+etKYGwJOvuACv0RN/MYaQDUbmHu4JsBxe08Trb6towBvB4eyrV+F93DkKnD'
    'PBr5fn4XzfUXGp+U62Sxoez+hRbtLxGpoSJfLwW6OIJ78jHV3rPY2we7ddJ4s9HM5eIhfqVIK1kdPI7x8124fxjJH0PaiLgf4w/4'
    'rns/EEWVZDPb6AbYu/FIAs/lT3g531ADF5KfAe5w0qlzrr/PCaApEsfy42ZH9endCqxcYHq83TEFJrEqxTy7I12LWTRYIMBBJDlJ'
    '1j4vFovilg2W0w+fBlYUVFCvkXjuQK2hTkSyMxFjzNh7bDaaZD+1HWCYM429URkGB3tSBBjr+LYS2tOhLRGce67PUsekwKAdeCNu'
    'imKmbCsVRiqIgVq2iaqLtXjdktR7HVvCcknhDnb3I/WqIB3ToQ7b/6GbRxkj7FHyR/0LzEDUIJ3NAlR0xj6JZH8S0QNPLtjoobsa'
    '1XelhTZcJ3KpTzDvrWq5EVyo7iYaJOsGZIO2HYc0wCh+4Sf+C25uEjcVMPwfvE9GPMqqzJZox5JUX6O5nUryJdSQiQvW+DRjdX7Q'
    '7IbFphEAik8kVwFMUwUSScwCA7nmbDF3jOc+64GsBoLSx+X7lD6E6VH5jICWP3j/JsRKjwp5bYnGzyK9EAvWuDKuWQ9PStNKqjKg'
    'skFTA8PxeVVQcIUcFPeIxhzxb+W6k+sNCPA6jtHaQj5BIM/iFotmpGWLWFY9RgtaQvwRmah8b/rlaW+5S2FSoEl5RuzeZAnKER1z'
    'LHLjcX0oxwcTZrWthSpL2Z7YxklJkIZbw+ABxDJ6wi0jWaXNRmToT9MbhUzuVtOlph2zM+6Yn1ZwQFouUVt9CbXm1RzoUzJxrDCr'
    'ulGq1FzDhAGaHIMcdHqBe3aqUZipy/oqXqZ3JCHvYAwcm3TH3UZQKUR1dBBIfow2GEPTNY06fAH/PA+4rn2sh96GzSCpoomOQ+jU'
    'pIrsH/aUYT2DiCMSnfLleDiadEoOPssxFrDAqphGH2Q6JqmS7JqrStkFhT3hUbcxynWMZZkIhjCJuu9egAi0etrMguHT7s4DaTDZ'
    'NEJ3vV1F7Jo/2Tt0xm8ewB+8H9aLxf5NMU0v1ouUXJXE40CZXi3WGARNfjdRzkEfWoOGgM5ND30rjQlYV2IxH2jeu16J8gapFHQD'
    'kMzoBa/F4v7Eu4Aui5IiXohz8NJm5adCf6tAhSsv5HwqVqzUih8G5DdNQFcp0TOtsCK1We+FN3j93AYilAGaLFHywvthUBXzGscY'
    'DAfwLGsbI9IIuRNvv/2S2wHKWYo0p6gk/N//kHsNOaw8tM4SnAdURkg/b0d0UzW851aBthcfqQmmzxT/rdWYRXqJysIJIzm5QiJl'
    'rsm/DxoOZThJD0hFpU2RaHhsMg+a9hi/jLNR9mJ4bHpas7KqowqEVT57An3xwr/bQmJkeeAYYmoG4C6RoQzDOOamTNn2tyvBwotg'
    'paMaBFaJu+ksu+D/FLIhPX/ca+DaQszuaf/LkGyExYG4q1GoBEyu/+IFXMbsCVJIOKgL4lqdnhPGX8QoxFgeV+tlC8qYGAYLqP9j'
    'wI4im2+aN0RdYasSOsYR67mTPGg9xdgtlMwAClpLQOOh+WuoSLLkFhFQdQalWfyK5jOrxRLZA0K6YmCoUWowhPpWiUGHzU+HZccf'
    '0TWQGCoAQ7OWzI2+fNn0L4LBqfCaRALp0TvaaPatG7CXQafaMOycxdBknGujYylqWONV/EBR1daSxaXfVSDGCDox9Xb6PLG1W9rZ'
    'TZWx0Tm50WgMG09xGJ0ctw4R4bYOqD5Wl6mDUi2Khu2Zl9UY0l/u8oJsjrw8AkV0DsZUhDtGs7Y//k/pIgX+NsOoJkFbqOhsv65O'
    'kGMzdnAn/Ea66ktsCX+XQsdBZRcL0XLFy/gTpWcDq7mJMdzDydtk1/jPi7zF3GgbCFUHvQvEY2rvWBLYM+RoN5ZahSOJWWVRXOyy'
    'vrKtFRmV/JH+YNjI7AVVCWv5aj2fg9GL9XR2L2FQF+wGc3v1sdAYlYbNnmqKN6LwXNa5FQLq5SqmYy+o+ycVdCS7C3wVTY5xuJrJ'
    'UAQUVkMVRFAy1HUMYJYW6ZSbMBYOxgc3hKUigjFUiAODIpNU2H1HQabOWGAODDEAzChiK1yYoNwxw0Q1nZhhLrAhcBiLmdGkz2IV'
    't79DZL3Ss+XJHg/MrQKNqroAei+Aa5aeDlx1tusMMN4a5c2TtkxzUL5wwc+I7ppgUBlO5ggzJZ3CDjXdMcp0dYVnfw5OvE4UYjPD'
    'O0UiNuC/VTTiU6fLiEncbcoYcTtFKjLynxavyKj9LQIWDS3FGcqLx5OWWR40PueI9B1ety9fHj05RnBzXCCNLDLiCIlJ7BAo2I4H'
    'bEgm7CG4TXGB1InewEAl65vAGIZ/Ynyglsa/a2igbuX3iQpsVe8ICNwY28dLWgb4dfYRXJF+K3QTOvYcfkXcH1Wdp6vqqmjH55KM'
    'M1iPHeCqpZekWZNGdeQusYGnhOi2gnsla2iV3xqsm21G1a8Na3TowH2xV+RGBwQ16hdh6oVkHs6AxE635TcqOTJm6skxiRzY+D5u'
    'GJstASju0NR8x9mkUSB1LKMJEUZvIwq2xKjGt6zGTKwDcta5KvIsch3uUE4zitIOoOzs9MnWm0FhR7pQpGdacZZfF2tJFC3jLVtB'
    'h5JYN4RftsMgY9td9YQwTJfo3hKUKWP8donKNM4ibgnP7KpHu4Zo6qVNgZpKnotVh2Cbs+pSU8Z/+2M0OVLdtUo7NTehlzu307f8'
    'VHQlKUw6nMGOq6Qqvyqcsn3Sr8NoNkRbbs9eYCUtYB2Hzl7j/vmuOQu+PlVBJ0MBUv2hmWDAmVNga0qBQ04agIGgncQB3aQBKitC'
    'K9+BOycCCVYcb8zP25MbaA+uEZKqTKDOwWL7HO2W5AW75S3Q29RGFI6dvoA9Dqs0wzjaTiKDTWlxtqcqUCkKwJjYb7IUdBIUNLkJ'
    'zMQATSIAS3nYeOzfTidgnL6fA1Fe8SH7CD0vZMb50rSzMxRw7gaK6YAJOHx9TGkQ2nkCwgjR4AZyBzsbRa0G+Zi2NO6mMCkZEm9S'
    'AW+Ct3laordsllWrokovFiZJkRcpTynkAvcFCjO9yj8tY0BW0VC/PmVA39lw1wYeRjHUtE8rT4Sf4HY1rEDQzXtX3O+fX6DFEczs'
    'AogZCh+/tTwj/fkG2vX/98o+wOKWHRibzncbNBCYZ5Gdh72b770Hvt1GRbtwwzl5hfom87Rh5bY/2haXnHiqHa93M/TD3QiTjp1L'
    'DouH+NJuXgLL82AEdD3tLLopGH7jtBBtBybHGyk3punC3JwSgvn905I9WJkdGm73u2d3cEW8/gKdytEh3ES9Qq9JqqKioTS+Usid'
    'zAg/5ZTeCNSgFCWwFrsLkZYUW0K+GzviNS0vAW+V2BICu0PMqwxrJcgUbKAEWfvlGlR9VYAexAw/bI1tfWoE66aUevKkWFrJM2Nb'
    '0+rBn34YjXSE1A87ZuFrHrHDZ99+PP3p83/TUNizv/yENb15JWbzdJhCfeLtbPju7eHxu1evh+JCvH73Rryez9/OBfQA8+N9+Pnz'
    '2c80nAYJq7VGHCWT6wTLmhG3n//nx9Nvv8enw/P8rx++//nHU3wI/L6TXCAxe6LF/ba+4/dPY081zRI0v9qTuktLOlslspI/gxjB'
    'MT34HVelPxq8ffM68vyt7kaABECjWX+T95CgjQxzeO4GtddAugTNUxNy+6PTtYn33qOuOz923V+NaFMBgmCm18m9QG9kK9sMt7h9'
    'yEYPdgB29Ij6Pcf97nxx7+5BLwatxvvh3JigWK+E9vzsfvst07oBBuMCQyg0EdVpde2bO30grUr2B7R2GGVFkosPWAcJSAOerZer'
    'Sm4nYvxggsepWD2XhmNRVnEA5B75Iz+MKPYyydNcbjcNZKhfGA6uxN0suxQVeShM278sijryJLeW5VTP4BMQPhkr+Nuw2bXpAN8J'
    '7KXHvMR0BTTf/GlxJSN4xJ2YrnEhInfBT6wCNbuYWl8xi9s6Fr5EjkI/mgBv0kQaSmFhPl3PUjQ20ps0W6BlFHQjty0NQ2CwIHsI'
    'AFGl+kznY78jvQNxR6/q+5Vg/OU5FIrYWibRn4H4EIvFwG3A4wapFCe2CU/fdrPfMbemVhwk2jmFp2Wug4LEeVMocUg76+73JVqG'
    'ZHIY8eaccw0Zn8yKNPKVB2sfT/KY2QV9btof8YgiI1noqLHd/Yu0ymhTa5kh7F9+6nr9fZjDNXTEHz3oRHxtm1ptW+DnsBUFIgXQ'
    'o6NWOlE/IseVjyaVNCdG7NXiV8r5M5JSLeJkZMv0TkEPjw9gECQdrAou0Y/kj4jl4OiB8PyR2wHxaBjThoIVsNbM61D+DVEFKW5z'
    'URpE0BjsLhtdzl2k0i7FXtcej7ye8ADNYb/sls8PAU27vH/Tv3eppTeC1wsbv/U9EOoc+Gk18NBFptM+A+mKi6K4pvV1dv8Z0epJ'
    'pPJZVEycONhiFf8+tnx/vkCh3GvanscB4dsyo9PqOt4BAwVkYF6TDRuTXwxse6a9X/hF7QnbcRE0t30+lorQHnu7poZT7AuLtfMz'
    '00GJvpALs5gVG2LKQzWCnpMWLtR+pzbGO+nFFH+jUEaF5w4SG+e8/+0Cx3zvqUGM/Eh2WOWXnkT+P/xI5l3p9cEbwpzKmxEEc2+1'
    'vlhk1ZXc5Y94a/T5c5jnxayznUBLfkDmX3dDTx6iflBTMJJV+lSnP+KqNbZHLkzrlh830ZaZZ8a1r8END9YrQHMQRrwXCeowIEtO'
    'S+Rt2LJwmuV6R1vMgtAdwaNQqXczI5k21Y+QS8T+R7EE7unhqkbyuy/W3hTmmE5j1dLyLoVbSBvZkDsb4+7u6N7aYSg+GfJEotJy'
    '8Tf50/Sw1CYOE0kHbXNy5DSMXVlFFh/XjIDhe1mB3D6ymYEs1c8OmgyPO6xs1kkAEtVcFxhVgpTgT57CBqT40Aa+XB3SN+Rc9QCM'
    'mqXZYhvIITbMcgrdKLL1bpijnQ1ruLvSGNt44nPjqmX0Wd/mYaRGMoKnR+dS9X+0adHjLPkjZGxzPLQGNjJKnZ6oF/8f3qXIMWAf'
    'lmhe2cXlt4S+JWZlvQxA7ujZrY96uapeIFrcRD7uI1dXyAMks479VVFluNXROAzAhgPKn6ntpWSdSwKJDX08jqXuuu08bYNC8q+q'
    'yQsj3oIhw88ZdPcE1rKZvRgsRuv+ysPYGOtb+ErUMwzbQYyaMWg9vK2DT34XwMPlXA9ce8mRqmDLNrLUubvowvKt+E7F9Z+Msk5l'
    '9t0S6quV6hS5FXV7R7UPbTLCCtaRlhkscU9QAgSV/ierWTnhnVfKaJRKV3+HZ9m+cqbSJ3RF7upWTrc55c5c551WS9oXBy2DjDBC'
    'TBOWr+wtsnYenRo2msXSz6HVbOBDEQU36OsCDrz3sTykCT+GnBBGnYqkLgw48cvuOvdfciiRYTYb8vLs0+661G3tQSqW0fjXsMnQ'
    'kdlUa11NAL0Oj2dzcAME7TQksjGW2/irgzcF8Sze5lZy4ILWNHpyZbp4RiN2nnYSjGs1HGV7OAQtXYUk+TeSETQz8p9VsQtlT2R0'
    '21cuzYMcAtktpkPXyXWoaeM4hx9pMz5GK35XBiSH0jS+KyvabVA9VT+LN5pU/UZR+wYV7fNhKelUh9rkjeekR0aP4HGiTmizqaxs'
    'oqjDJyPrHpytctS8gydq35BjXL8TOdMEt9/Ki3oe3QhuRmiYdGS3Oi4N6kG3FLeYHI9XFh7BkUfAadMLGamdc7inqqesNnv7IjZG'
    'E/amMHjKytuNUJXpucQrPHh7ZMd1R28QRfJ5K1m0rwB5mr5gnQxZPm297o4KV/3dm6o0gKVQPIv71Iltq9u41cWjEIRfoVjYAozY'
    'Oct71fXdRL6l0ONZShb/dFIgIJEfZejyRt/dwfv45n0sZT2d+TPa3yjudxH71tU3fZLfKbzjB80eRlolMHuHhhb7ec23wKx0TA65'
    'fC2KD3vQ1VILYj+dw3wxK/JftPplcC1H7pNFL4uL27eiSSNotFNvZEGajlZxVw+c9nK/UxSd25fILrp3fikd6LfVgn9gDdhxLYj0'
    'uxJL5sNTXU2YaWM347OFxYjpYicNajNb3CQFIk1pcYtqusYubRxubEc6rAzLmaVi3HEReG2D312g3ymwsRd99rUSkKs1n2VSluVu'
    'ljf7QYx9VWO1t6dNrudErnW5yO058EfmJGyTagY/GLWm6THK6Gqz+DCMnuorFXdTsaq9U/ojA1bgXZ9fkeLvlbZBD/HcfyBWDaXC'
    'QZIgl0+Sx5H3AC8euzZhhmFHFLgz9xQ0cJkkWeJ5jcRvIsErUcYqUmnwbXm5Ri/8Gb0PZqKalhn1OE6SWTFNktAsiKmUklSWCfz9'
    'fdwG9iN5d+DMRFBfAd4+259lldyXxUWaTvl0JMj2EpPlrXXAHZStYlkX/cHadPgjKjr4PKC9cv7FDSRmAz2RYWbYio4N+3xbeBxs'
    'jJvMlGeoWPABk5QTeb50pjRuotyi1r14xVyfLuhmNO5EjLWSJC4pKExlTGyAkH8+OYyMfiu4PDffDnIdBdaXOPFXpCi0g5S+P/3p'
    '06nOBXh8dAAwP374rN+8Ojyw8gViONOfP3z7I2UCfPO6iZJQym2AbnAraqMUA+AdM1R5yvO98bf7/5Hu/33y4h/jg/13k3+M/8/5'
    '+e35eTXBrMxU2rgaaoV79KWxyyGzPneys+HxH8yQ1izGQGaDVslxMaMxsOR2giidM1rmfZPZFQaUmiKwQK1Ua1ZONzt3Gn+eqKQ0'
    '/94kuMny6WKNKYI56/UaM9tQuqslTN0FKKg5kPFiH1/ozGdeLpC0MZi11Hm1ZJ4Ryp1DiZz33q/S2TeYmPbAyB+VRPh/mSYCEdWf'
    'js5MKxupZK22mollkKxop1bNuysNW+je6aSy0t9Ffe9RjOjbGKExjzOeY6M35nVRrgTgahLNdGgAi9QRXGYzwgRvWgO61QuFGtKh'
    '0OzlPHq/DlF3Xuy1M0GNeUw12wu1G4MN/iZq/oi6cBR2go9qUdSV3UwXnWM3hsf9EhnPQpVjlcV5Enr73vHrDRKctlnGTU7nyabD'
    'qkblh0+u/HDHymXO5821T3oUfQvtXaiJM8WzTg9TLGadyVejUFmmZd6cTdNLmbr76uEk1DvUwnd/SE4ncxZiAOpdxPQTUX9lqlZ9'
    'gD8Tuy4vnQST0/e1G38wyQiab9GIWoaq990laNmF/2jqO2zXd/jk+gws4doj+TFvpc7GNNm9yRL7s2PL1IVsuRNyDGp5GlPAlMuS'
    'DeBPOuyCvTOnuJfYbkJHT8yLq1zMYzzGBXQtV43H9UTetpeTzU0xPTVZ+S3aax/FJHqNDI4fyfKsGUwXoPZ5f6LzVXk++CvFwpkp'
    'hpMED84mSYBp4fA+iVl91Tqcv6ZYjIGGM3GCmeTwgC50Eir/Mb0X5U/wGHAtFuCX6xsJh4dTSgaSLXrPvVc2PAeDuuFt2PmcQT/h'
    'xnaOsVbWynJ0L9rUF/r459Mffw66cAjQ7QepYgqrML90wpmRemci9EvEMw/zap0UwWjYqxSMp7sB/R0fwJpVvzEj72Hk7Q/DAW3l'
    'rIpKBEN4F3YVDiJ9hfNAz1JwF4aD6dUaNMFXVJNzef8devbDoALDFOz2WVEnnA2qTtJaXigVyCEAbVdgwK6rdMEmlN0zNaAkraDt'
    '9rK+817oaQ7+3k1Kob/P51S2Iebv0fjvEjNbKd9RdzzdV4+sHJkNEbnByxXaCvIGPE5meKKyalbEmYBhUUJWkTOTMq906Vk1tPIk'
    'ScSvDp+0fkjDI+I9XV6IGak1Zn02PEcjUI7jVpnjo54ClM2q6qwP5glqKZhPvQvRKL3behoehBuWBTFnydpMlF1RjmiJGlL1qlBR'
    'gxp90EpjSDB6oYRdatKUZmAkuArHI1wHkxZ5ybiWT9irPio7kxEhytAG1fgGUyGCEU6G8cy4vocFBzqqkapI0Rp4PxVAk3i8DiNN'
    '8b1M5El5gbfTGxNaO2XkVkLDHTeLZt72kAzf3LIjMA/TAh4e9ED3cPXhwVcQyrUiFBhaQF9xQsfQ9OHEFKg3Co66akACl3rVgvyi'
    'IAkDDeRRO7+OYi7Aw69BGnxBENyml3kigbV6L8kPMqi+lHVwLUmTSa2PNIMbqErXTQlPMQEUVIbVt1sY7pTQtALlTGWmkDly+3OY'
    'mkg2lVzECBU1M/CiW4Fy8LamRqaFXyNdxxJS5e3U4oayV+KNPgg2EF8CQ6e5LArkhQQyuMnELRTDi5laGo9xic75Hjkf0IhHNYvL'
    'cabOyISi7VyEQjWJGm5/x4SW82JdqoqwHwP0wQxDV3UyFKTSNQK4E4I8vnT3DaXgMKvlNzo9bbu7t0Uic26rHgUSm5cpJkJD8Ypo'
    'a+jO+8brQODsNRDOkTDbSS4yHkyJ96rw/longS9XL1VQTd/wNjgE1v9a1fu4A11y8pMucVaNM0yFDzz2ptxt/FG2fkzaj0yCL51T'
    'ZoYl3vCHygmvTMaqhtAgVzSDqL5OAlm5MaYV9IfzPVkrIlH+jDp0h3U9bkwa2Rx+Vno9JaPgnWul2FPOTY2p/FIbJX/G3RE8woVk'
    'tSUFoS60JQmhtyhjvMcEVE8rE+HB0LwzE+pjy9m4Rc70+0yUS5JiilpzzwmXZF3YLzC5B8siL+oiz6aBea05ZZ1etYOieLcSDfCM'
    '4/Xxk5ltlHfkYgfVcTlTMW0REpdBU0wPynJyJU3Crcb1a3u2ehI5OZjpXZuRmkmTSCnfsCCtOLSnJYnamJnp67IzecMBiOHtCZpc'
    'SZratwwYk25dn3a+x3NLV57Rr8gGaPbAeWX2UECnnMzjg4WMfIw2lMx83oYCIa5Xu1HC9DG3b49tqTQ9xPIH79vFbXpfKX84xkR4'
    'KLGshO/6RBAdcaLM3gPHxY1NfgbkgTrrGd0Y8PIl3roI2l4o92SRo8TAZ9p5xFH3kdeoKBEDIz8CPb2lmR+F2goMWhanS91IpL4R'
    '3GDm7UkT0WGyiN2UkKevm69IrPYbZTLbda2414nMJ83q6APdQkk3q6Ln8hFJ06UFEmqDyrgEhj8or6gWnJJs9VY7tIEKTNPo+Hoy'
    'tvWdifdN7NHWF/m55CYATeLL9lbATNwYb4zUXObKH7kZwlgtb7m5h+1u5fIuHVIPji89lA8tPYn36tdlmU3Xi/VSMxPZlRaswTtY'
    'brdUO2IZCTseNAh6lA5bcA2BEBAoIKsB3m8A+k9zhrBLS23tTizSFRijCWfQNzQ8W8R6+x7oK6/apZml4nl5YrC4x9+qXJMAfm6e'
    '2nqgOhXeHGxpGdKfSXegvb15didm+xfrGeAHaLakU3d6r7tCmxkvy7mkCwZEOuMoSdxeKzM0mihPsOtiPyszkWZ/PbmJ2ueo3Olb'
    'WozsaxLeSO1O+nn0BoKxh9t3SO+x6+7DsAnex9aaIp37JLmHdOLYuqLLfTGIhv13uBo5od5I+teUM9nY9448Ps0fOauzlgjH6OxY'
    'LzE6d7UqgoiYO1Zn+mcir9mD7+md6/jPxizdXbTLqMWY1XO8xy7cpJ8j+zLwb2c/7qYJ2iVF0E68C89J3I1bLyfNDUAyTbIOiGyt'
    '5p7IKmIZjASbYCZWQ4acKpsyPVRhF271hdha1TTd+ogIJ5YoLBaNl3gvEX+U6hVWYkKhATr0wzbJUT61a8a8d/jlfO9DLvb5/hFg'
    'nBXVCLxJHq2ui/sWXyK6JkDgzxUpNvtqT0pdmwNgihVxY+r0HHsH5W06leE9rJrrupDMChWcguwvy2+yCq9nQJ6X1Xbd/4sRy+GH'
    'noxyqry2/KXqTfF7QpnrjFY40hX7qG8Fslv6SWQyL7s6A0hXtqECqnOTtq+7xJ0E7JXM24y0shCX6fSe1wVIuowijtpSxIgO44tx'
    'VYCYvh03xdu5nx4flq7aoV7ne/tyminC5n4l4rNUOzExUgubWbliulR2BJR2SpSIu1odg+UQQXWIWYfoaYWHosC4aYM1GG8HKOcE'
    'H52lil/gZAB29lSQB8UjyqggM3DMkbFIx419S/Fi6oTjvvJnLwArMK8FGxN89qw5hlB5VUbulRp3TojAoLxgnai1u9tKNLZzHrFd'
    '7s7tz85FMV5QiPOFbsnzhUN7wi27v+utujo7EzQN0qB11yZGEcSU8qoTziUDWJVmoO4PjDvRCJE7cMsOTlA3Cz5sunzSFeRl3rBD'
    'XTVDvMxTxKqL0oMW44WgfC3jV9/9yem1HXd+XkfNJvuo58A5XSMeN030Hdlp3RH5VfEOT4t2xwY/65vbKMp0RoOCBvgub7rGO60l'
    'BKWswG9XoDz86849oLs1jPsk9Q2a+Fvhju6RFPdxfK0E07P4xqUywUSoq3JwcQ3M2yGjm4gakxEXdLkkvQg7d40yVTvuD8UGNNfU'
    'lCSvVp14/2K8A/Hm00W7qBhhW0xkz+JXfDehHFiuizhPi3RzJJkXlurrOpERgL3lqQvI0ZzhEEyS2GaqbOMW2UrmgkEGrxpv5+rH'
    'flvXY+pwnUcay9ZEsB/pDih1OoI1EiujtxR6T8ni7Tj/qMJl/wm5vH9FWm2d9fv4fawH9T4+fH38dfnAjSzcEllQ9WCA9XXyg6P0'
    '7eYIb51MoVW0JTc3wXxFgu4UZGl2Y+YK3SlF96YE3HRgytA2KJ2XTMNtDMXIxd3uvG0GNmSmfxl5uu3c2xsTdS9kW47c25RIuJ2u'
    'G12sWottnCx+xKdMrxAW+Gw8eBfR4dLmxZtOBCWywPiBs0RRBrKV4cax0pFF9NjO3c3vqPNNPu/Hf3bC7Z1Sen1Fvu0th5BVJjmZ'
    '1+v/X3Jt+9yZkV572j0p/Vum1f6902n/2qzNyUZ5YY1nC5N72hk8dxaKypDQ6gTA75PG2TyBsR0Jo9bFevipe7HeyfTafR1fc9ce'
    'CTPXhW6Yvchx1d6GzEuObGquxdm7MFWOCddtbevcd7a0+916Rt6z3yJJOTnt4pZ9ZcQX7JBOXt1w9iuzyG8U0JzIHcvaadyt9O1d'
    'NybM25gzU06aDf+dbxtUsnGXKwfpOq6Dk86Fg9PrnS4anF53bxN0XDm4wwWDIydlNgdzdRbIZVZRJr+vvDzwadcGOq8IJJy1rxLc'
    'emvgiTXnjsx/zSlgvi0Pfc0R+T5bkSI0XnWNHBLqw+MJajLx0riJwLjU9uERG79Rn9s33j48hu47B41r26L2/YP2oWX7PkJ83uH6'
    'QTMlCKqAOXfPdSeyH0ZNQhADlvweft+xGCNtSAy/N1avk4o0kJsrd2QeiQOaK56RcGNz2+vkzjhqlN3afEsgiaqdbwnEW/tM4iL2'
    'QW9hdXeIYcNFfa2b+Ha/w08uJ4tqtt7ap/JB/sa3G2a61v6kktZleeZF147OPFyP2IHSvjCaw6yuJ+bV9MPQ2Gnv3lzdWrvh41fe'
    'Q6hMiCuQsbRjN+pc50l3ovXe5hnbl6dZe6J80H+57Z7Ob8DiIvt0ufmuzW9iOnVrX29Irai72OKYRGaTkdm1XfgH76f2zggeNMUd'
    'IQ9ll0dBCGRN4zECwUlSKNkuX7IIehBepN2plll0/xQ3N322p/pJ6QwdCYkMLNDepCunR7RkCRJ2ro10WXuIuhHzg13zqzrujSQX'
    'rhx6/32NhIoYL3qvWpdGEh71JY50w2H0loTM87eRqRf1XbFIQn73OxWpPftiRfM087abFXe5S5H3U3a4SPHk97s/Ud2ciH9exMMt'
    'lxtuXVDLhp+cdC9TPHFRUTe/la/CTfBh6b4i8XyP3bSMRHUzot27R9pJBmbrZDtOOTx5fInBOnsu2QSteldiMUNv5IODP/VXeL7X'
    'ztax9/j/ADItMOQ='
)
launch()
